# Адаптер «правки + короткие размышления» (Colab A100 80 ГБ)

По логам MiMo в 121 из 149 эпизодов кончилось время (во всех 52 `no_patch`), а ~70% текста рассуждений --
перед `read_file`/`run_command`, часто это пересказ только что прочитанного кода. Идея как у ThinkingCap:
модель учится останавливаться, когда информации уже хватает на следующий вызов.

Один адаптер (в отправке может быть только один) учится на двух видах пар:
* `pref_edit` (188 пар, как у `pref_edit_v1`, давшего 0.15 -> 0.20): правильный / сломанный `edit_file`;
* `short_think` (`train.build_pref_short`): тот же разведочный вызов с рассуждением, обрезанным на первом абзаце,
  где уже названа цель вызова (chosen), против исходного длинного (rejected).

Обучение -- DPO с reference (`--ref`): без него длинный вариант проигрывает уже из-за числа токенов.

В Drive (`MyDrive/gemma_adapters/`): отправка-основа (`submissions/v5.zip`), `pref_edit.jsonl`, логи с рассуждениями:
`traj_think.zip` (~300 пар) и `mimo_p1_full_150.zip` (~60 пар); оба zip лежат в корне проекта на ноутбуке.
Задачи проверочного набора `eval/mimo50.json` из пар исключаются, поэтому адаптер можно честно сравнивать на mimo50.

Порядок: **0** параметры, **1** карта, **2** код, **3** Drive и пары, **4** модель, **5** обучение (vLLM не запускать),
**6** сохранение, **7** архив отправки.

## 0. Параметры (правьте только здесь)

In [ ]:
SUBMISSION_ZIP = "submissions/v5.zip"        # основа: промпт v3 и его настройки (путь внутри gemma_adapters)
MAX_TIME_MINUTES = 5.5                       # 8 мин при 60 вызовах не влезает в лимит Kaggle (v7); перезаписать max_time_minutes в eval_config (None -- оставить)
PAIRS = "pref_edit.jsonl"                    # пары правок (188)
# логи (папки или zip; берутся все подпапки с results.jsonl). eval_mimo50 сюда НЕ класть: задачи проверочного
# набора eval/mimo50.json исключаются из пар в любом случае, иначе сравнение на mimo50 нечестное
LOGS = ["traj_think.zip", "mimo_p1_full_150.zip"]
ADAPTER = "pref_short_v1"
SHORT_PER_EPISODE = 2                        # пар short_think с одного эпизода
EPOCHS = 1
EDIT_REPEAT = 2                              # пары pref_edit -- 2 раза за эпоху: как у pref_edit_v1 (2 эпохи)
MAX_LEN = 8192

## 1. Карта (нужна A100 80 ГБ, память свободна)

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
name, total, used = [x.strip() for x in out.split(",")]
assert int(total) >= 79000, f"нужна карта на 80 ГБ, а здесь {total} МиБ: смените среду выполнения"
assert int(used) < 2000, f"занято {used} МиБ: остановите vLLM (!pkill -f vllm) или перезапустите среду"

## 2. Код и зависимости

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
# torch и transformers в Colab уже есть; не обновляем их, чтобы не сломать совместимость
!pip install -q compressed-tensors bitsandbytes accelerate safetensors huggingface_hub kagglehub pyyaml

## 3. Drive, основа отправки и пары
Системное сообщение во всех парах -- промпт основы. В выводе: сколько пар каждого вида и какая доля текста
рассуждения остаётся в коротком варианте.

In [ ]:
import glob, json, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
for p in (SUBMISSION_ZIP, PAIRS):
    assert os.path.exists(f"{DRIVE}/{p}"), f"нет {DRIVE}/{p}"

SUB_DIR = "/content/submission_src"
shutil.rmtree(SUB_DIR, ignore_errors=True)
zipfile.ZipFile(f"{DRIVE}/{SUBMISSION_ZIP}").extractall(SUB_DIR)
root = next(d for d, _, files in os.walk(SUB_DIR) if "agent.yaml" in files)   # архив бывает с папкой верхнего уровня
if root != SUB_DIR:
    tmp = SUB_DIR + "_tmp"; shutil.move(root, tmp); shutil.rmtree(SUB_DIR); shutil.move(tmp, SUB_DIR)
SYSTEM_FILE = f"{SUB_DIR}/prompts/system.md"
system = open(SYSTEM_FILE, encoding="utf-8").read()
print(open(f"{SUB_DIR}/eval_config.yaml").read())
print("промпт:", len(system), "символов")

# логи: zip распаковываем в /content/logs, в папках ищем все подпапки с results.jsonl и messages/
runs = []
for item in LOGS:
    path = f"{DRIVE}/{item}"
    if not os.path.exists(path):
        print("пропуск, нет:", path); continue
    if path.endswith(".zip"):
        dst = "/content/logs/" + os.path.basename(path)[:-4]
        if not os.path.exists(dst):
            zipfile.ZipFile(path).extractall(dst)
        path = dst
    runs += sorted(os.path.dirname(p) for p in glob.glob(f"{path}/**/results.jsonl", recursive=True)
                   if os.path.isdir(os.path.join(os.path.dirname(p), "messages")))
print("прогонов с логами:", len(runs), *runs, sep="\n  ")
assert runs, "нет ни одного прогона с messages/"

os.makedirs("data", exist_ok=True)
SHORT = f"data/{ADAPTER}_short.jsonl"
EVAL_IDS = json.load(open("eval/mimo50.json"))["ids"]          # проверочный набор: из обучения исключаем
!python -m train.build_pref_short --runs {" ".join(runs)} --system {SYSTEM_FILE} --out {SHORT} --max-per-episode {SHORT_PER_EPISODE} --exclude {" ".join(EVAL_IDS)}

DATA = f"data/{ADAPTER}.jsonl"
n_edit = n_short = 0
with open(DATA, "w", encoding="utf-8") as out:
    for line in open(f"{DRIVE}/{PAIRS}", encoding="utf-8"):
        if not line.strip():
            continue
        rec = json.loads(line)
        for side in ("chosen", "rejected"):
            assert rec[side][0]["role"] == "system", "в паре нет системного сообщения"
            rec[side][0]["content"] = system
        out.write(json.dumps(rec, ensure_ascii=False) + "\n"); n_edit += 1
    for line in open(SHORT, encoding="utf-8"):
        if line.strip():
            out.write(line if line.endswith("\n") else line + "\n"); n_short += 1
print(f"пар: pref_edit {n_edit}, short_think {n_short} -> {DATA}")
assert n_edit >= 150 and n_short >= 50
leak = {json.loads(l).get("task") for l in open(DATA, encoding="utf-8") if l.strip()} & set(EVAL_IDS)
assert not leak, f"в парах задачи проверочного набора mimo50: {sorted(leak)[:5]}"
print("задач из eval/mimo50.json в парах: 0")

## 4. Базовая модель в `/content/model` (если уже есть, ничего не качает)

In [ ]:
import getpass
MODEL_DIR = "/content/model"
HF_REPO = "google/gemma-4-31b-it-qat-w4a16-ct"
KAGGLE_MODEL = "google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct"
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download(HF_REPO, local_dir=MODEL_DIR, token=hf_token)
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download(KAGGLE_MODEL)
        !cp -r "{path}"/. {MODEL_DIR}/
os.environ["GEMMA_MODEL_DIR"] = MODEL_DIR
!du -sh {MODEL_DIR}

## 5. Обучение (DPO с reference; ~5 ч; vLLM не запускать)
Пары правок идут 2 раза за эпоху (`EDIT_REPEAT`), короткие размышления -- 1 раз: правки модель видит столько же,
сколько `pref_edit_v1` за его 2 эпохи.

В логе: `пар: N ... по видам` (сколько влезло в MAX_LEN), `reference посчитан`, затем `eval` **по видам пар**
(`real_fix`, `synthetic`, `short_think`, `все`):
* `до обучения` -- с reference разность ровно 0, доля 0.00: это норма;
* `в конце` -- главное: у `real_fix` и `synthetic` доля должна дойти до ~1.0 (у `pref_edit_v1` было 0.57 -> 1.00).
  Если правки заметно ниже (< 0.9), а `short_think` высоко -- короткие пары перетянули обучение: поставьте
  `EDIT_REPEAT = 3` и повторите 0, 3, 5;
* `pref-loss` вниз от 0.69.

При `CUDA out of memory` -- `MAX_LEN = 6144` в ячейке 0, повторить 0 и 5.

In [ ]:
LOG = f"/content/{ADAPTER}.log"
!python -m train.pref_lora --data {DATA} --name {ADAPTER} --decompress --ref --epochs {EPOCHS} --max-len {MAX_LEN} --repeat real_fix={EDIT_REPEAT} synthetic={EDIT_REPEAT} 2>&1 | tee {LOG}

## 6. Сохранить адаптер, пары и лог в Drive

In [ ]:
assert os.path.exists(f"adapters/{ADAPTER}/adapter_model.safetensors"), "адаптер не сохранился, смотрите лог"
!cp -r adapters/{ADAPTER} {DRIVE}/ && cp {LOG} {DATA} {DRIVE}/
!ls -la {DRIVE}/{ADAPTER}
!grep -E "пар:|eval|reference" {LOG}

## 7. Архив отправки: основа + `adapter:` + файлы адаптера (+ время)
Результат: `gemma_adapters/upload_{ADAPTER}/submission.zip`.

In [ ]:
import re
OUT_DIR = f"/content/upload_{ADAPTER}"
shutil.rmtree(OUT_DIR, ignore_errors=True)
shutil.copytree(SUB_DIR, OUT_DIR)
shutil.rmtree(f"{OUT_DIR}/adapters", ignore_errors=True)
os.makedirs(f"{OUT_DIR}/adapters/{ADAPTER}")
for f in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy(f"adapters/{ADAPTER}/{f}", f"{OUT_DIR}/adapters/{ADAPTER}/{f}")
agent = open(f"{OUT_DIR}/agent.yaml", encoding="utf-8").read()
agent = re.sub(r"(?m)^adapter:.*\n?", "", agent)
agent = re.sub(r"(?m)^(model:.*\n)", rf"\1adapter: {ADAPTER}\n", agent, count=1)
assert f"adapter: {ADAPTER}" in agent, "в agent.yaml нет строки model:"
open(f"{OUT_DIR}/agent.yaml", "w", encoding="utf-8").write(agent)
if MAX_TIME_MINUTES is not None:
    cfg = open(f"{OUT_DIR}/eval_config.yaml", encoding="utf-8").read()
    cfg, n = re.subn(r"(?m)^(\s*max_time_minutes:).*$", rf"\g<1> {MAX_TIME_MINUTES}", cfg)
    assert n == 1, "в eval_config.yaml нет max_time_minutes"
    open(f"{OUT_DIR}/eval_config.yaml", "w", encoding="utf-8").write(cfg)
print(agent); print(open(f"{OUT_DIR}/eval_config.yaml").read())
size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(OUT_DIR) for f in fs)
print(f"распакованный размер: {size / 2**30:.3f} ГиБ")
assert size < 3 * 2**30
ZIP = f"/content/submission_{ADAPTER}.zip"
if os.path.exists(ZIP):
    os.remove(ZIP)
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for d, _, fs in os.walk(OUT_DIR):
        for f in fs:
            p = os.path.join(d, f)
            z.write(p, os.path.relpath(p, OUT_DIR))
print(*sorted(zipfile.ZipFile(ZIP).namelist()), sep="\n")
os.makedirs(f"{DRIVE}/upload_{ADAPTER}", exist_ok=True)
shutil.copy(ZIP, f"{DRIVE}/upload_{ADAPTER}/submission.zip")
print("готово:", f"{DRIVE}/upload_{ADAPTER}/submission.zip")